<a href="https://colab.research.google.com/github/sheramir/Stock_Research_Agent/blob/main/Fetch_Market_Symbols.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fetch Stocks and ETFs Names

In [4]:
!pip -q install yahoo_fin requests_html

In [10]:
import pandas as pd
import yfinance as yf
import yahoo_fin.stock_info as si

import requests
from io import StringIO

# Set the column width to display long text fully
pd.set_option('display.max_colwidth', None)

## Retrieve NASDAQ stocks and ETF's

In [60]:
# Retrieve Nasdaq stocks list

nasdaq = si.tickers_nasdaq(include_company_data=True)

# Convert the result to a DataFrame for easier searching
nasdaq_df = pd.DataFrame(nasdaq)

# Make ETF column Type column
nasdaq_df['ETF'] = nasdaq_df['ETF'].replace({'N': 'Stock', 'Y': 'ETF'})
nasdaq_df = nasdaq_df.rename(columns={'ETF': 'Type'})

# Remove unnecessary columns
nasdaq_df = nasdaq_df.rename(columns={'Security Name': 'Name'})
nasdaq_df = nasdaq_df[['Symbol', 'Name', 'Type']]

# Remove missing values
nasdaq_df = nasdaq_df.dropna()

nasdaq_df


,Symbol,Name,Type
0,AACB,Artius II Acquisition Inc. - Class A Ordinary Shares,Stock
1,AACBR,Artius II Acquisition Inc. - Rights,Stock
2,AACBU,Artius II Acquisition Inc. - Units,Stock
3,AACG,"ATA Creativity Global - American Depositary Shares, each representing two common shares",Stock
4,AADR,AdvisorShares Dorsey Wright ADR ETF,ETF
...,...,...,...
4831,ZVSA,"ZyVersa Therapeutics, Inc. - Common Stock",Stock
4836,ZYBT,Zhengye Biotechnology Holding Limited - Ordinary Shares,Stock
4837,ZYME,Zymeworks Inc. - Common Stock,Stock
4838,ZYXI,"Zynex, Inc. - Common Stock",Stock


## Retrieve S&P500 stocks

In [61]:
sp500_df = pd.read_html("https://en.wikipedia.org/wiki/List_of_S%26P_500_companies")[0]

# Remove unnecessary columns
sp500_df = sp500_df.rename(columns={'Security': 'Name'})
sp500_df = sp500_df[['Symbol', 'Name']]
sp500_df['Type'] = 'Stock'

sp500_df = sp500_df.dropna()

sp500_df

,Symbol,Name,Type
0,MMM,3M,Stock
1,AOS,A. O. Smith,Stock
2,ABT,Abbott Laboratories,Stock
3,ABBV,AbbVie,Stock
4,ACN,Accenture,Stock
...,...,...,...
498,XYL,Xylem Inc.,Stock
499,YUM,Yum! Brands,Stock
500,ZBRA,Zebra Technologies,Stock
501,ZBH,Zimmer Biomet,Stock


## Retrieve ETFs

In [67]:
# Use predefined ETF list. Generated using AI prompt
etf_list = {
    'SPY': 'SPDR S&P 500 ETF Trust - Tracks the S&P 500 Index',
    'IVV': 'iShares Core S&P 500 ETF - Tracks the S&P 500 Index',
    'VOO': 'Vanguard S&P 500 ETF - Tracks the S&P 500 Index',
    'QQQ': 'Invesco QQQ Trust - Tracks the NASDAQ-100 Index',
    'VTI': 'Vanguard Total Stock Market ETF - Tracks the CRSP US Total Market Index',
    'IWM': 'iShares Russell 2000 ETF - Tracks the Russell 2000 Index',
    'DIA': 'SPDR Dow Jones Industrial Average ETF Trust - Tracks the Dow Jones Industrial Average',
    'VEA': 'Vanguard FTSE Developed Markets ETF - Tracks the FTSE Developed All Cap ex US Index',
    'AGG': 'iShares Core U.S. Aggregate Bond ETF - Tracks the Bloomberg U.S. Aggregate Bond Index',
    'GLD': 'SPDR Gold Shares - Tracks the price of gold bullion',
    'VTV': 'Vanguard Value ETF - Tracks the CRSP US Large Cap Value Index',
    'VO': 'Vanguard Mid-Cap ETF - Tracks the CRSP US Mid Cap Index',
    'VB': 'Vanguard Small-Cap ETF - Tracks the CRSP US Small Cap Index',
    'VUG': 'Vanguard Growth ETF - Tracks the CRSP US Large Cap Growth Index',
    'BND': 'Vanguard Total Bond Market ETF - Tracks the Bloomberg U.S. Aggregate Float Adjusted Index',
    'EFA': 'iShares MSCI EAFE ETF - Tracks the MSCI EAFE Index (developed markets outside US & Canada)',
    'XLK': 'Technology Select Sector SPDR Fund - Tracks the technology sector of the S&P 500',
    'XLF': 'Financial Select Sector SPDR Fund - Tracks the financial sector of the S&P 500',
    'ARKK': 'ARK Innovation ETF - Actively managed, focuses on disruptive innovation companies',
    'TIP': 'iShares TIPS Bond ETF - Tracks the Bloomberg U.S. Treasury Inflation-Protected Securities Index'
}

etf_df = pd.DataFrame(list(etf_list.items()), columns=['Symbol', 'Name'])
etf_df['Type'] = 'ETF'

etf_df

,Symbol,Name,Type
0,SPY,SPDR S&P 500 ETF Trust - Tracks the S&P 500 Index,ETF
1,IVV,iShares Core S&P 500 ETF - Tracks the S&P 500 Index,ETF
2,VOO,Vanguard S&P 500 ETF - Tracks the S&P 500 Index,ETF
3,QQQ,Invesco QQQ Trust - Tracks the NASDAQ-100 Index,ETF
4,VTI,Vanguard Total Stock Market ETF - Tracks the CRSP US Total Market Index,ETF
5,IWM,iShares Russell 2000 ETF - Tracks the Russell 2000 Index,ETF
6,DIA,SPDR Dow Jones Industrial Average ETF Trust - Tracks the Dow Jones Industrial Average,ETF
7,VEA,Vanguard FTSE Developed Markets ETF - Tracks the FTSE Developed All Cap ex US Index,ETF
8,AGG,iShares Core U.S. Aggregate Bond ETF - Tracks the Bloomberg U.S. Aggregate Bond Index,ETF
9,GLD,SPDR Gold Shares - Tracks the price of gold bullion,ETF


## Merge tables

In [68]:
# prompt: merge s&p_df with nasdaq_df without repeating same symbols

import pandas as pd
# Concatenate the three DataFrames
combined_df = pd.concat([sp500_df, nasdaq_df, etf_df])

# Remove duplicate symbols, keeping the first occurrence
combined_df = combined_df.drop_duplicates(subset=['Symbol'], keep='first')

# Remove rows with missing values
combined_df = combined_df.dropna()

# Display the merged DataFrame
combined_df


,Symbol,Name,Type
0,MMM,3M,Stock
1,AOS,A. O. Smith,Stock
2,ABT,Abbott Laboratories,Stock
3,ABBV,AbbVie,Stock
4,ACN,Accenture,Stock
...,...,...,...
15,EFA,iShares MSCI EAFE ETF - Tracks the MSCI EAFE Index (developed markets outside US & Canada),ETF
16,XLK,Technology Select Sector SPDR Fund - Tracks the technology sector of the S&P 500,ETF
17,XLF,Financial Select Sector SPDR Fund - Tracks the financial sector of the S&P 500,ETF
18,ARKK,"ARK Innovation ETF - Actively managed, focuses on disruptive innovation companies",ETF


In [72]:
# Sort table by 'Symbol'
combined_df = combined_df.sort_values(by='Symbol')
combined_df

,Symbol,Name,Type
9,A,Agilent Technologies,Stock
0,AACB,Artius II Acquisition Inc. - Class A Ordinary Shares,Stock
1,AACBR,Artius II Acquisition Inc. - Rights,Stock
2,AACBU,Artius II Acquisition Inc. - Units,Stock
3,AACG,"ATA Creativity Global - American Depositary Shares, each representing two common shares",Stock
...,...,...,...
4831,ZVSA,"ZyVersa Therapeutics, Inc. - Common Stock",Stock
4836,ZYBT,Zhengye Biotechnology Holding Limited - Ordinary Shares,Stock
4837,ZYME,Zymeworks Inc. - Common Stock,Stock
4838,ZYXI,"Zynex, Inc. - Common Stock",Stock


In [71]:
# Search for string in the DataFrame (any column or in specific column)

def find_symbol(df, search_term, column=None):
    # Search both 'Symbol' and 'Name' columns for the input symbol
    if column is None: # Search in any column
      return df[df.apply(lambda row: row.astype(str).str.contains(search_term, case=False).any(), axis=1)]
    else: # Search in specified column
      return df[df[column].str.contains(search_term, case=False)]

# Example usage:
search_term = "SPY"  # Replace with the desired search term
results = find_symbol(combined_df, search_term, 'Symbol')
results


,Symbol,Name,Type
4057,SPYQ,Tradr 2X Long SPY Quarterly ETF,ETF
4341,TSPY,TappAlpha SPY Growth & Daily Income ETF,ETF
4781,YSPY,GraniteShares YieldBOOST SPY ETF,ETF
0,SPY,SPDR S&P 500 ETF Trust - Tracks the S&P 500 Index,ETF


## Save to file

In [73]:
# Save merged table to csv file
combined_df.to_csv('stocks_etfs.csv', index=False)